
KMNIST, MLP, регуляризация и оптимизация

In [27]:
import os
import json
import csv
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

ROOT = '.'
ARTIFACTS_DIR = os.path.join(ROOT, 'artifacts')
FIGURES_DIR = os.path.join(ARTIFACTS_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

cpu


In [29]:
import tarfile

transform = transforms.ToTensor()
dataset_name = 'CIFAR10'

data_root = './data'
local_tar = './data/cifar-10-python.tar'
extracted_dir = os.path.join(data_root, 'cifar-10-batches-py')

if os.path.exists(local_tar) and not os.path.exists(extracted_dir):
    with tarfile.open(local_tar, 'r') as tar:
        tar.extractall(path=data_root)

full_train = datasets.CIFAR10(root=data_root, train=True, download=False, transform=transform)
test_dataset = datasets.CIFAR10(root=data_root, train=False, download=False, transform=transform)

train_size = int(0.8 * len(full_train))
val_size = len(full_train) - train_size
split_generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(full_train, [train_size, val_size], generator=split_generator)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

x_batch, y_batch = next(iter(train_loader))
input_dim = int(np.prod(x_batch.shape[1:]))
num_classes = len(full_train.classes)
print('dataset:', dataset_name)
print('x shape:', x_batch.shape)
print('y shape:', y_batch.shape)
print('x range:', float(x_batch.min()), float(x_batch.max()))
print('input_dim:', input_dim)
print('num_classes:', num_classes)

dataset: CIFAR10
x shape: torch.Size([128, 3, 32, 32])
y shape: torch.Size([128])
x range: 0.0 1.0
input_dim: 3072
num_classes: 10


In [30]:
class MLP(nn.Module):
    def __init__(self, hidden_sizes=(512, 256), dropout=0.0, batchnorm=False):
        super().__init__()
        layers = [nn.Flatten()]
        in_features = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(in_features, h))
            if batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_features = h
        layers.append(nn.Linear(in_features, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [31]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, total_correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            total_correct += (logits.argmax(dim=1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, total_correct / total

def fit_model(model, train_loader, val_loader, optimizer, max_epochs=10, early_stopping=False, patience=4):
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc = -1.0
    best_loss = float('inf')
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    wait = 0
    for epoch in range(max_epochs):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc = evaluate(model, val_loader, criterion)
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)
        improved = va_acc > best_acc
        if improved:
            best_acc = va_acc
            best_loss = va_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            wait = 0
        else:
            wait += 1
        if early_stopping and wait >= patience:
            break
    model.load_state_dict(best_state)
    return model, history, best_acc, best_loss, best_epoch

In [32]:
runs = []
histories = {}

def model_summary_str(hidden_sizes, dropout, batchnorm):
    return f'hidden={list(hidden_sizes)};activation=ReLU;dropout={dropout};batchnorm={batchnorm}'

def run_experiment(experiment_id, hidden_sizes, dropout, batchnorm, opt_name, lr, momentum=0.0, weight_decay=0.0, epochs=10, early_stopping=False, patience=4):
    torch.manual_seed(SEED)
    model = MLP(hidden_sizes=hidden_sizes, dropout=dropout, batchnorm=batchnorm).to(device)
    if opt_name == 'Adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    model, history, best_val_acc, best_val_loss, best_epoch = fit_model(
        model,
        train_loader,
        val_loader,
        optimizer,
        max_epochs=epochs,
        early_stopping=early_stopping,
        patience=patience,
    )
    histories[experiment_id] = history
    run = {
        'experiment_id': experiment_id,
        'dataset': dataset_name,
        'seed': SEED,
        'model_summary': model_summary_str(hidden_sizes, dropout, batchnorm),
        'optimizer': opt_name,
        'lr': lr,
        'momentum': momentum if opt_name == 'SGD' else 0,
        'weight_decay': weight_decay,
        'epochs_trained': best_epoch,
        'best_val_accuracy': best_val_acc,
        'best_val_loss': best_val_loss,
    }
    runs.append(run)
    return model, run

In [ ]:
base_hidden = (512, 256)
base_lr = 1e-3

model_e1, run_e1 = run_experiment('E1', base_hidden, 0.0, False, 'Adam', base_lr, epochs=10)
model_e2, run_e2 = run_experiment('E2', base_hidden, 0.3, False, 'Adam', base_lr, epochs=10)
model_e3, run_e3 = run_experiment('E3', base_hidden, 0.0, True, 'Adam', base_lr, epochs=10)

best_from = 'E2' if run_e2['best_val_accuracy'] >= run_e3['best_val_accuracy'] else 'E3'
if best_from == 'E2':
    e4_dropout, e4_batchnorm = 0.3, False
else:
    e4_dropout, e4_batchnorm = 0.0, True

best_model, run_e4 = run_experiment(
    'E4',
    base_hidden,
    e4_dropout,
    e4_batchnorm,
    'Adam',
    base_lr,
    epochs=20,
    early_stopping=True,
    patience=4,
)

torch.save(best_model.state_dict(), os.path.join(ARTIFACTS_DIR, 'best_model.pt'))

best_config = {
    'dataset': dataset_name,
    'seed': SEED,
    'hidden_sizes': list(base_hidden),
    'dropout': e4_dropout,
    'batchnorm': e4_batchnorm,
    'optimizer': 'Adam',
    'lr': base_lr,
    'early_stopping_patience': 4,
}
with open(os.path.join(ARTIFACTS_DIR, 'best_config.json'), 'w', encoding='utf-8') as f:
    json.dump(best_config, f, ensure_ascii=False, indent=2)

hist_e4 = histories['E4']
epochs_axis = np.arange(1, len(hist_e4['train_loss']) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs_axis, hist_e4['train_loss'], label='train_loss')
plt.plot(epochs_axis, hist_e4['val_loss'], label='val_loss')
plt.plot(epochs_axis, hist_e4['train_acc'], label='train_acc')
plt.plot(epochs_axis, hist_e4['val_acc'], label='val_acc')
plt.xlabel('epoch')
plt.ylabel('value')
plt.title('E4 curves')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'curves_best.png'), dpi=150)
plt.show()

print('best_from:', best_from)
print('best_val_accuracy:', run_e4['best_val_accuracy'])

In [14]:
model_o1, run_o1 = run_experiment('O1', base_hidden, e4_dropout, e4_batchnorm, 'Adam', 1e-1, epochs=6)
model_o2, run_o2 = run_experiment('O2', base_hidden, e4_dropout, e4_batchnorm, 'Adam', 1e-5, epochs=6)
model_o3, run_o3 = run_experiment('O3', base_hidden, e4_dropout, e4_batchnorm, 'SGD', 5e-3, momentum=0.9, weight_decay=1e-4, epochs=12)

hist_o1 = histories['O1']
hist_o2 = histories['O2']
epochs_o1 = np.arange(1, len(hist_o1['train_loss']) + 1)
epochs_o2 = np.arange(1, len(hist_o2['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(epochs_o1, hist_o1['train_loss'], label='train_loss')
axes[0].plot(epochs_o1, hist_o1['val_loss'], label='val_loss')
axes[0].set_title('O1 lr=1e-1')
axes[0].set_xlabel('epoch')
axes[0].legend()

axes[1].plot(epochs_o2, hist_o2['train_loss'], label='train_loss')
axes[1].plot(epochs_o2, hist_o2['val_loss'], label='val_loss')
axes[1].set_title('O2 lr=1e-5')
axes[1].set_xlabel('epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'curves_lr_extremes.png'), dpi=150)
plt.show()

print('O1 best_val_accuracy:', run_o1['best_val_accuracy'])
print('O2 best_val_accuracy:', run_o2['best_val_accuracy'])
print('O3 best_val_accuracy:', run_o3['best_val_accuracy'])

NameError: name 'e4_dropout' is not defined

In [15]:
criterion = nn.CrossEntropyLoss()
test_loss, test_acc = evaluate(best_model, test_loader, criterion)

best_config['test_accuracy'] = float(test_acc)
best_config['test_loss'] = float(test_loss)
with open(os.path.join(ARTIFACTS_DIR, 'best_config.json'), 'w', encoding='utf-8') as f:
    json.dump(best_config, f, ensure_ascii=False, indent=2)

fieldnames = [
    'experiment_id',
    'dataset',
    'seed',
    'model_summary',
    'optimizer',
    'lr',
    'momentum',
    'weight_decay',
    'epochs_trained',
    'best_val_accuracy',
    'best_val_loss',
]
with open(os.path.join(ARTIFACTS_DIR, 'runs.csv'), 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(runs)

print('best_from:', best_from)
print('E4 val acc:', run_e4['best_val_accuracy'])
print('test acc:', test_acc)
for r in runs:
    print(r['experiment_id'], round(r['best_val_accuracy'], 4), round(r['best_val_loss'], 4))

NameError: name 'best_model' is not defined